<a href="https://colab.research.google.com/github/aravinth-xuno/fraud-detection-notebook/blob/main/Data_Transformation_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import pandas as pd
import numpy as np

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
train_df = pd.read_csv("/content/drive/Shareddrives/transaction-training-set/dataset/training_dataset_high_velocity_users.csv", low_memory=False)

In [17]:
train_df.shape

(15440, 29)

In [18]:
train_df['INITIATED_AT'] = pd.to_datetime(train_df['INITIATED_AT'], format="ISO8601", utc=True)

In [19]:
train_df = train_df.sort_values(by=['USER_ID', 'INITIATED_AT']).reset_index(drop=True)

In [20]:
grouped = train_df.groupby('USER_ID')

In [21]:
train_df['cnt_1h'] = grouped.rolling('1h', on='INITIATED_AT')['TRANSACTION_KEY'].count().values - 1

In [22]:
train_df['last_1hour_count'] = train_df['cnt_1h']

In [23]:
train_df['cnt_total_25h'] = grouped.rolling('25h', on='INITIATED_AT')['TRANSACTION_KEY'].count().values - 1

In [24]:
train_df['last_24hour_count'] = grouped.rolling('24h', on='INITIATED_AT')['TRANSACTION_KEY'].count().values - 1

In [25]:
train_df['last_24hour_count_shifted'] = train_df['cnt_total_25h'] - train_df['cnt_1h']

In [37]:
train_df['last_30days_count'] = grouped.rolling('30D', on='INITIATED_AT')['TRANSACTION_KEY'].count().values - 1

In [ ]:
train_df['cnt_total_31d'] = grouped.rolling('31D', on='INITIATED_AT')['TRANSACTION_KEY'].count().values - 1

In [28]:
train_df['last_30days_count_shifted'] = train_df['cnt_total_31d'] - train_df['last_24hour_count']

In [29]:
train_df.drop(columns=['cnt_total_25h', 'cnt_total_31d'], inplace=True)

In [30]:
train_df['ratio_1hr_24hr'] = (train_df['last_1hour_count']+1) / (train_df['last_24hour_count_shifted']+1)

In [31]:
train_df['log_ratio_1hr_24hr'] = np.log1p(train_df['ratio_1hr_24hr'])

In [32]:
train_df['ratio_24hr_30day'] = (train_df['last_24hour_count']+1) / (train_df['last_30days_count_shifted']+1)

In [33]:
train_df['log_ratio_24hr_30day'] = np.log1p(train_df['ratio_24hr_30day'])

In [34]:
transformed_columns = [
    "last_1hour_count",
    "last_24hour_count",
    "last_30days_count",
    "last_24hour_count_shifted",
    "last_30days_count_shifted",
    "ratio_1hr_24hr",
    "log_ratio_1hr_24hr",
    "ratio_24hr_30day",
    "log_ratio_24hr_30day"
]

In [38]:
features = [
    "last_1hour_count",
    "last_24hour_count",
    "last_30days_count",
    "ratio_1hr_24hr",
    "ratio_24hr_30day"
]

In [39]:
train_df[features].head()

,last_1hour_count,last_24hour_count,last_30days_count,ratio_1hr_24hr,ratio_24hr_30day
0,0.0,0.0,0.0,1.0,1.0
1,0.0,0.0,1.0,1.0,0.5
2,1.0,1.0,2.0,2.0,1.0
3,2.0,2.0,3.0,3.0,1.5
4,3.0,3.0,4.0,4.0,2.0


In [41]:
train_df[features].describe()

,last_1hour_count,last_24hour_count,last_30days_count,ratio_1hr_24hr,ratio_24hr_30day
count,15440.000000,15440.000000,15440.000000,15440.000000,15440.000000
mean,0.082772,0.128303,1.456671,1.058413,0.707331
std,0.340711,0.438070,2.438664,0.354729,0.465613
min,0.000000,0.000000,0.000000,0.090909,0.025000
25%,0.000000,0.000000,0.000000,1.000000,0.333333
50%,0.000000,0.000000,1.000000,1.000000,0.500000
75%,0.000000,0.000000,2.000000,1.000000,1.000000
max,7.000000,9.000000,42.000000,8.000000,10.000000


In [40]:
train_df.to_csv("/content/drive/Shareddrives/transaction-training-set/dataset/transformed_dataset_high_velocity_users_features.csv", index=False)